In [ ]:
# IMPORTS
from memo import memo, domain
from math import exp
import jax
import numpy as np
import jax.numpy as jnp
from jax.scipy.stats import beta as jax_beta
from scipy.stats import beta as sp_beta_dist
import matplotlib.pyplot as plt
import seaborn as sns

# GLOBALS
INFO, ACTION = 0, 1
CAN, IMP, NULL = 0, 1, 2
goals = [INFO, ACTION]
U = [CAN, IMP, NULL]

# COSTS
c_can, c_imp = .1, .7 # utterance costs for speaker
c_effort = .1 # physical cost for actioner
w_cost = 0.0 # willingness cost for actioner
c_act = c_effort + w_cost

prior_ga = .5 # uniform prior over goals

mu_vals = np.linspace(0.001, .999, 50)   # grid over ability
theta_vals = np.linspace(0.01, 0.99, 20) # threshold values
kappa = 20   # tightness of beta_agg dist (a+b)
alpha = 5    # softmax rationality param

# CONTEXT-DEPENDENT PRIOR OVER ABILITY
# The mu prior is the actioner's prior over the speaker's ability belief,
# shaped by common ground about the context.
#
# ctx_mu:  expected ability in this context
# ctx_k:   certainty about ability (higher = more concentrated)
#
# Uniform:          ctx_mu=0.5,  ctx_k=2  → Beta(1,1) (matches original model)
# "benchpress 150": ctx_mu=0.5,  ctx_k=2  → diffuse (genuinely uncertain)
# "call me later":  ctx_mu=0.95, ctx_k=20 → peaked near 1 (obviously able)

def mu_prior(mu, ctx_mu, ctx_k):
    """Beta prior over speaker's ability belief, shaped by context."""
    mu_c = jnp.clip(mu, .01, .99)
    a = ctx_mu * ctx_k
    b = (1 - ctx_mu) * ctx_k
    return jax_beta.pdf(mu_c, a, b)

In [ ]:
## A0 -- literal actioner
# CAN and IMP share the same truth-conditional core (both depend on ability)
# but have different speech act types:
#   CAN = question about ability → elicits an ANSWER (resolves INFO QUD)
#   IMP = directive to act       → elicits COMPLIANCE (satisfies ACTION goal)
#
# Acting entails answering (doing X proves you can), but not vice versa.
#
# f1, f2, f3 are circumstantial preconditions (Kratzer modal base):
# proximity, motor ability, hands free, etc.
f1, f2, f3 = .05, .99, .9
f_agg = float(np.prod([f1, f2, f3]))  # NOTE: unused (mu plays this role)

# P(f_agg > theta | speaker belief = mu)
def p_yes(mu, t):
    mu_c = jnp.clip(mu, .01, .99)
    return 1.0 - jax_beta.cdf(t, mu_c * kappa, (1 - mu_c) * kappa)

@jax.jit
def A0_answer(u, mu, t):
    """Prob of answering 'yes' — resolves INFO QUD.
    Only CAN (a question) elicits a literal answer."""
    p = p_yes(mu, t)
    return jnp.where(u == CAN, p, 0.)

@jax.jit
def A0_comply(u, mu, t):
    """Prob of taking the action — satisfies ACTION goal.
    Only IMP (a directive) elicits literal compliance.
    (CAN's indirect-request reading emerges at S2 via goal inference.)"""
    p = p_yes(mu, t)
    return jnp.where(u == IMP, p, 0.)

print(f"f_agg={f_agg:.3f}  (unused — mu plays this role)")
print(f"A0_answer(can, ...) = {float(A0_answer(CAN, mu_vals[25], theta_vals[1])):.3f}")
print(f"A0_answer(imp, ...) = {float(A0_answer(IMP, mu_vals[25], theta_vals[1])):.3f}")
print(f"A0_comply(can, ...) = {float(A0_comply(CAN, mu_vals[25], theta_vals[1])):.3f}")
print(f"A0_comply(imp, ...) = {float(A0_comply(IMP, mu_vals[25], theta_vals[1])):.3f}")

In [ ]:
## S1
u_label = {CAN: 'can', IMP: 'imp', NULL: 'null'}
goal_label = {INFO: 'info', ACTION: 'action'}

@jax.jit
def eu_s1(u, g, mu, t):
    ans = A0_answer(u, mu, t)     # CAN→p, IMP→0, NULL→0
    act = A0_comply(u, mu, t)     # CAN→0, IMP→p, NULL→0
    v = 4 * ans * (1 - ans)       # info value from answer (0 for IMP/NULL)
    cost = jnp.where(u == CAN, c_can, jnp.where(u == IMP, c_imp, 0.))
    return jnp.where(g == ACTION, act, v) - cost

# P(u | g, mu, t)
@memo
def S1[_u: U, _g: goals, _mu: mu_vals, _t: theta_vals](alpha):
    speaker: knows(_g, _mu, _t)
    speaker: chooses(u in U, wpp=exp(alpha * eu_s1(u, _g, _mu, _t)))
    return Pr[speaker.u == _u]

s1_out = S1(alpha)
ti_mid = int(len(theta_vals) // 2)
for gi, g in enumerate(goals):
    probs = {u_label[u]: round(float(s1_out[ui, gi, 25, ti_mid]), 3) for ui, u in enumerate(U)}
    print(f"S1(g={goal_label[g]}, mu=.5, t={theta_vals[ti_mid]:.2f}) = {probs}")

In [ ]:
## A1

@jax.jit
def goal_prior(g, prior_ga):
    return jnp.where(g == 1, prior_ga, 1 - prior_ga)

@memo
def A1[_g: goals](prior_ga, alpha, ctx_mu, ctx_k):
    a1: knows(_g)
    a1: thinks[
        spk: given(g in goals, wpp=goal_prior(g, prior_ga)),
        spk: given(mu in mu_vals, wpp=mu_prior(mu, ctx_mu, ctx_k)),
        spk: given(t in theta_vals, wpp=1),
        spk: chooses(u in U, wpp=S1[u, g, mu, t](alpha))
    ]
    a1: observes_that[spk.u == 0]
    return a1[Pr[spk.g == _g]]

def a1_act(prior_ga, ctx_mu=0.5, ctx_k=2.0):
    p_ga = float(A1(prior_ga=prior_ga, alpha=alpha, ctx_mu=ctx_mu, ctx_k=ctx_k)[ACTION])
    eu_do = 1 - c_act
    eu_ans = 1 - p_ga
    probs = jnp.exp(alpha * jnp.array([eu_do, eu_ans]))
    return float(probs[0] / probs.sum())

print("A1 goal inference (uniform mu prior):")
for p in [.3, .5, .85]:
    p_ga = float(A1(prior_ga=p, alpha=alpha, ctx_mu=0.5, ctx_k=2.0)[ACTION])
    print(f"  prior={p}: P(g=action|u=can) = {p_ga:.3f}, P(act) = {a1_act(p):.3f}")

In [ ]:
## S2
# S2 knows A1 does goal inference, so CAN now has action utility:
# A1 may infer g=ACTION from CAN and comply → p_a1_do

@jax.jit
def eu_s2(u, g, mu, t, p_a1_do):
    ans = A0_answer(u, mu, t)
    act = A0_comply(u, mu, t)
    v = 4 * ans * (1 - ans)
    # CAN gets indirect action value via A1's goal inference
    act_s2 = jnp.where(u == CAN, p_a1_do, act)
    cost = jnp.where(u == CAN, c_can, jnp.where(u == IMP, c_imp, 0.))
    return jnp.where(g == ACTION, act_s2, v) - cost

@memo
def S2[_u: U, _g: goals, _mu: mu_vals, _t: theta_vals](p_a1_do, alpha):
    speaker: knows(_g, _mu, _t)
    speaker: chooses(u in U, wpp=exp(alpha * eu_s2(u, _g, _mu, _t, p_a1_do)))
    return Pr[speaker.u == _u]

s2_out = S2(p_a1_do=a1_act(.5), alpha=alpha)
for gi, g in enumerate(goals):
    probs = {u_label[u]: round(float(s2_out[ui, gi, 25, ti_mid]), 3) for ui, u in enumerate(U)}
    print(f"S2(g={goal_label[g]}, mu=.5, t={theta_vals[ti_mid]:.2f}) = {probs}")

In [ ]:
## A2

@memo
def A2[_g: goals](prior_ga, p_a1_do, alpha, ctx_mu, ctx_k):
    a2: knows(_g)
    a2: thinks[
        spk: given(g in goals, wpp=goal_prior(g, prior_ga)),
        spk: given(mu in mu_vals, wpp=mu_prior(mu, ctx_mu, ctx_k)),
        spk: given(t in theta_vals, wpp=1),
        spk: chooses(u in U, wpp=S2[u, g, mu, t](p_a1_do, alpha))
    ]
    a2: observes_that[spk.u == 0]
    return a2[Pr[spk.g == _g]]

def a2_act(prior_ga, ctx_mu=0.5, ctx_k=2.0):
    p_a1 = a1_act(prior_ga, ctx_mu, ctx_k)
    p_ga = float(A2(prior_ga=prior_ga, p_a1_do=p_a1, alpha=alpha,
                     ctx_mu=ctx_mu, ctx_k=ctx_k)[ACTION])
    eu_do = 1 - c_act
    eu_ans = 1 - p_ga
    probs = jnp.exp(alpha * jnp.array([eu_do, eu_ans]))
    return float(probs[0] / probs.sum())

print("A2 goal inference (uniform mu prior):")
for p in [.3, .5, .85]:
    p_ga = float(A2(prior_ga=p, p_a1_do=a1_act(p), alpha=alpha,
                     ctx_mu=0.5, ctx_k=2.0)[ACTION])
    print(f"  prior={p}: P(g=action|u=can) = {p_ga:.3f}, P(act) = {a2_act(p):.3f}")

In [ ]:
## plots: A1 vs A2

def act_at(prior_ga, c, ctx_mu=0.5, ctx_k=2.0):
    """P(do X) for A1 and A2 at given prior, cost, and context"""
    eu_do = 1 - c - w_cost

    p_ga1   = float(A1(prior_ga=prior_ga, alpha=alpha, ctx_mu=ctx_mu, ctx_k=ctx_k)[ACTION])
    eu_ans1 = 1 - p_ga1
    p1      = jnp.exp(alpha * jnp.array([eu_do, eu_ans1]))
    p_a1    = float(p1[0] / p1.sum())

    p_ga2   = float(A2(prior_ga=prior_ga, p_a1_do=p_a1, alpha=alpha,
                        ctx_mu=ctx_mu, ctx_k=ctx_k)[ACTION])
    eu_ans2 = 1 - p_ga2
    p2      = jnp.exp(alpha * jnp.array([eu_do, eu_ans2]))
    p_a2    = float(p2[0] / p2.sum())

    return p_a1, p_a2

priors = np.linspace(0.01, 0.99, 60)
costs  = np.linspace(0.0,  0.95, 60)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

ax = axes[0]
a1v, a2v = zip(*[act_at(p, c_act) for p in priors])
ax.plot(priors, a1v, label='A1', color='steelblue')
ax.plot(priors, a2v, label='A2', color='tomato')
ax.axvline(prior_ga, color='gray', linestyle='--', linewidth=.8, label=f'current prior={prior_ga}')
ax.set_xlabel('prior P(g_action)')
ax.set_ylabel('P(act | u_can)')
ax.set_title('vs prior (uniform mu)')
ax.legend(); ax.set_ylim(0, 1)

ax = axes[1]
a1v, a2v = zip(*[act_at(prior_ga, c) for c in costs])
ax.plot(costs, a1v, label='A1', color='steelblue')
ax.plot(costs, a2v, label='A2', color='tomato')
ax.axvline(c_act, color='gray', linestyle='--', linewidth=.8, label=f'current C_act={c_act}')
ax.set_xlabel('c_act (c + w)')
ax.set_ylabel('P(act | u_can)')
ax.set_title('vs action cost (uniform mu)')
ax.legend(); ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
## plots: S1 vs S2
s2_out_curr = S2(p_a1_do=a1_act(prior_ga), alpha=alpha)

u_colors = {CAN: 'steelblue', IMP: 'tomato', NULL: 'gray'}
u_labels = {CAN: 'can', IMP: 'imp', NULL: 'null'}

fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharex=True, sharey=True)

for ri, (gi, glabel) in enumerate([(ACTION, 'g=action'), (INFO, 'g=info')]):
    for ci, (out, mlabel) in enumerate([(s1_out, 'S1'), (s2_out_curr, 'S2')]):
        ax = axes[ri, ci]
        for ui in U:
            ax.plot(mu_vals, out[ui, gi, :, ti_mid],
                    color=u_colors[ui], label=u_labels[ui], linewidth=2)
        ax.set_title(f'{mlabel} | {glabel}')
        ax.set_ylim(0, 1)
        if ri == 1: ax.set_xlabel('mu (speaker ability belief)')
        if ci == 0: ax.set_ylabel('P(utterance | g, mu)')
        ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
## Context comparison: "benchpress 150" vs "call me later"
# When ability is uncertain (benchpress), INFO value of CAN is high → literal question
# When ability is obvious (call me), INFO value → 0 → CAN only useful via ACTION at S2

contexts = {
    'benchpress 150':  (0.5,  2.0),   # diffuse: Beta(1,1) ≈ uniform
    'call me later':   (0.95, 20.0),  # peaked near 1: Beta(19,1)
    'pass the salt':   (0.99, 40.0),  # near-certain: Beta(39.6, 0.4)
}

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Panel 1: goal inference by context
ax = axes[0]
x = np.arange(len(contexts))
width = 0.35
a1_goals = [float(A1(prior_ga=0.5, alpha=alpha, ctx_mu=cm, ctx_k=ck)[ACTION])
             for cm, ck in contexts.values()]
a2_goals = [float(A2(prior_ga=0.5, p_a1_do=a1_act(0.5, cm, ck), alpha=alpha,
                      ctx_mu=cm, ctx_k=ck)[ACTION])
             for cm, ck in contexts.values()]
ax.bar(x - width/2, a1_goals, width, label='A1', color='steelblue', alpha=0.7, edgecolor='black', linewidth=0.5)
ax.bar(x + width/2, a2_goals, width, label='A2', color='tomato', alpha=0.7, edgecolor='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(contexts.keys(), fontsize=9)
ax.set_ylabel('P(g=action | u=can)')
ax.set_title('Goal inference by context')
ax.legend(); ax.set_ylim(0, 1)

# Panel 2: compliance by context
ax = axes[1]
a1_acts = [a1_act(0.5, cm, ck) for cm, ck in contexts.values()]
a2_acts = [a2_act(0.5, cm, ck) for cm, ck in contexts.values()]
ax.bar(x - width/2, a1_acts, width, label='A1', color='steelblue', alpha=0.7, edgecolor='black', linewidth=0.5)
ax.bar(x + width/2, a2_acts, width, label='A2', color='tomato', alpha=0.7, edgecolor='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(contexts.keys(), fontsize=9)
ax.set_ylabel('P(act | u=can)')
ax.set_title('Compliance by context')
ax.legend(); ax.set_ylim(0, 1)

# Panel 3: sweep ctx_mu, show goal inference at A1 and A2
ax = axes[2]
ctx_mus = np.linspace(0.1, 0.99, 30)
a1_sweep = []
a2_sweep = []
for cm in ctx_mus:
    g1 = float(A1(prior_ga=0.5, alpha=alpha, ctx_mu=cm, ctx_k=10.0)[ACTION])
    p_a1 = a1_act(0.5, cm, 10.0)
    g2 = float(A2(prior_ga=0.5, p_a1_do=p_a1, alpha=alpha, ctx_mu=cm, ctx_k=10.0)[ACTION])
    a1_sweep.append(g1)
    a2_sweep.append(g2)
ax.plot(ctx_mus, a1_sweep, label='A1', color='steelblue', linewidth=2)
ax.plot(ctx_mus, a2_sweep, label='A2', color='tomato', linewidth=2)
ax.set_xlabel('context ability (ctx_mu)')
ax.set_ylabel('P(g=action | u=can)')
ax.set_title('Goal inference vs context ability')
ax.legend(); ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()